In [1]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train_data=pd.read_csv("/content/drive/MyDrive/textsummarizer_transformers/samsum-train.csv")
val_data=pd.read_csv("/content/drive/MyDrive/textsummarizer_transformers/samsum-validation.csv")

In [4]:
#we can observe the data has symbols like emojis and html tags and all
#sample() method gives us the any random value sample(5) gives 5 random values like the random rows

train_data.sample(10)

,id,dialogue,summary
1271,13681456,Bob: What is the name of this thing that you w...,Bob will buy some transparent nail polish for ...
5707,13828023,"Emily: Hi Emma, I'm thinking of freshening my ...",Emily wants to change her haircut but she wasn...
9307,13717175,Amelia: I’ve just listened to No Doubt song Do...,Amelia heard the song Dont Speak on the radio ...
10668,13681964,"Alex: Hey, are you coming today?\r\nFilip: Yea...","Filip is running late today, whereas the profe..."
2948,13729585,Viera: Hey I saw you tried some catering. Whic...,Mary will try Speedy Lunch or Ligh box caterin...
5034,13865195,Pieter: could anybody turn on the heating?\nJe...,Maria will turn on the heating.
4012,13865233,"Marsha: Guys, are you back from Madagascar?\nT...","Tracy, Bob and Dominic came back from Madagasc..."
11895,13730571,Crosby: Do you know the rug in the hallway?\r\...,Crosby wants Marie to get the rug in the hallw...
12314,13727855,Viola: Hey!\r\nViola: I'm downtown and my head...,"Viola's headphones got destroyed, so she needs..."
12121,13730549,Liz: Have you watched The Little Drummer Girl ...,Tony didn't like The Little Drummer Girl.


In [5]:
# we will be training on 4k data nad validation on 500 as we have limited values

train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

# Data Pre-Processing

In [6]:
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ",text) #lines
    text=re.sub(r"\s+"," ",text) #spaces
    text=re.sub(r"<.*?>"," ",text) #html tags
    text=text.strip().lower() #removing trailing spaces and convert to lower case
    return text

In [7]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

# Tokenization

In [8]:
tokenizer=T5Tokenizer.from_pretrained("t5-small") #computationally good fo us

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [9]:
#t5tokenizer is for getting its vocabulary it has around 32k words

#raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True) #as summary should be in less words
    inputs["labels"]=targets["input_ids"] #token ids=>add to input as labels
    return inputs


In [10]:
train_dataset=train_data.apply(tokenize,axis=1).tolist() # input ids,attention mask(shows actual,paading values),labels ,,,more compatible with hugging face
val_dataset=val_data.apply(tokenize,axis=1).tolist()

# Working with our model

In [11]:
# NLP=>generation task
#as here generation is dependent on the inputs so it is conditonal generation
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
print(device)
model.to(device) #givint the device to model

cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [13]:
#defining the training arguments

training_args=TrainingArguments(
    output_dir='/content/drive/MyDrive/textsummarizer_transformers/results', #to create the model params
    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch", # after every epoch we do it

    warmup_steps=500 #starting learning rate is 0 =>lr default in 500 steps

)

In [14]:

#Trainer class is for transformers

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [15]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.646227,0.381357
2,0.396700,0.359929
3,0.373713,0.354569
4,0.361237,0.350261
5,0.355312,0.349290
6,0.351099,0.348931


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9140477193196614, metrics={'train_runtime': 1260.3653, 'train_samples_per_second': 19.042, 'train_steps_per_second': 2.38, 'total_flos': 3248203235328000.0, 'train_loss': 0.9140477193196614, 'epoch': 6.0})

In [16]:
model.save_pretrained("/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model") # to save the model for later use
tokenizer.save_pretrained("/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model/tokenizer.json')

In [17]:
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/textsummarizer_transformers/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [21]:
# test the core logic for summarization

def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue) #clean
  #tokenize
  #pt is pytorch tenors for hugging fce
  inputs=tokenizer(dialogue,padding="max_length",max_length=512,truncation=True,return_tensors="pt").to(device)
  model.to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4, #4 different answers generates andd comapres among them
      early_stopping=True #only 4 no need any other
  ) #generate the summarry=>token ids
  #token ids convert to summary
  summary=tokenizer.decode(targets[0],skip_special_tokens=True) #EOS,SEP
  return summary



In [22]:
test_dialogue="""Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him ðŸ™‚
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye"""
summary=summarize_dialogue(test_dialogue)
print(summary)

larry called larry last time she was at the park together. he called her last time they were at the park together.
